In [ ]:
import json
import os
from collections import Counter

def analyze_licenses(file_path):
    # Count how many models use each license
    license_counts = Counter()
    total_models = 0

    if not os.path.exists(file_path):
        print(f"Error: File {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            
            model = json.loads(line)
            total_models += 1
            
            # Extract the license field
            lic = model.get('license')
            
            if lic is None:
                license_counts['None/Unknown'] += 1
            elif isinstance(lic, list):
                # If it's a list, count every license in that list
                license_counts.update(lic)
            else:
                # If it's a string, just count the one
                license_counts[lic] += 1

    # Convert the Counter to a sorted list of unique licenses
    unique_licenses = sorted(list(license_counts.keys()))

    print("-" * 40)
    print(f"LICENSE ANALYSIS")
    print("-" * 40)
    print(f"Total Models Analyzed: {total_models}")
    print(f"Total Unique Licenses: {len(unique_licenses)}")
    print("-" * 40)
    
    # Print the most common licenses first
    print("Top Licenses by Frequency:")
    for lic, count in license_counts.most_common():
        print(f"{lic}: {count}")
    
    return unique_licenses

In [ ]:
# Configuration
DATA_DIR = 'clean_license_data'
FINAL_FILE = 'final_combined_licenses.jsonl'

path = os.path.join(DATA_DIR, FINAL_FILE)

unique_list = analyze_licenses(path)

## Cleaning up licenses and grouping for graphing
- License naming conventions are inconsistent (e.g. Apache, Apache-2, apache 2.0)
- Sometimes we may want to group licenses (e.g. llama 2, llama3)

In [ ]:
import pandas as pd
import os

# ==========================================
# 1. CLEANING LOGIC (Kept from previous versions)
# ==========================================
def standardize_license(lic):
    if not lic or not isinstance(lic, str):
        return "Unknown/Other"

    lic_low = lic.lower().strip()

    # Standard License Mapping
    standards = {
        'apache-2.0': 'Apache-2.0', 'apache-2': 'Apache-2.0', 'apache': 'Apache-2.0',
        'mit': 'MIT', 'modified-mit': 'MIT',
        'bsd-2-clause': 'BSD-2-Clause', 'bsd-3-clause': 'BSD-3-Clause',
        'gpl-3.0': 'GPL-3.0', 'agpl-3.0': 'AGPL-3.0',
        'cc-by-4.0': 'CC-BY-4.0', 'attribution 4.0 international.': 'CC-BY-4.0',
        'cc-by-nc-4.0': 'CC-BY-NC-4.0', 'license.cc-by-nc-4.0.txt': 'CC-BY-NC-4.0',
        'cc-by-nc-sa-4.0': 'CC-BY-NC-SA-4.0', 'cc-by-nc-nd-4.0': 'CC-BY-NC-ND-4.0',
    }
    if lic_low in standards: return standards[lic_low]

    # Family Grouping
    family_groups = {
        'llama': 'Llama-Community', 'qwen': 'Qwen-Community',
        'minimax': 'Minimax-Community', 'ltx': 'LTX-Community',
        'flux': 'Flux-NonCommercial', 'nvidia': 'NVIDIA-Model',
        'stabilityai': 'Stability/OpenRAIL', 'openrail': 'Stability/OpenRAIL',
        'openmdw': 'OpenMDW',
    }
    for keyword, group_name in family_groups.items():
        if keyword in lic_low: return group_name

    if any(x in lic_low for x in ['non-commercial', 'research-license', 'research-and-non']):
        return 'Research-NonCommercial'

    noise = ['unknown', 'other', 'none', 'model-license', 'license_source', 'more information needed']
    if any(n in lic_low for n in noise): return 'Unknown/Other'

    return lic.strip()

def process_license_field(lic_value):
    if isinstance(lic_value, list):
        cleaned_list = list(set([standardize_license(l) for l in lic_value]))
        return cleaned_list[0] if len(cleaned_list) == 1 else cleaned_list
    return standardize_license(lic_value)

# ==========================================
# 2. DATAFRAME PIPELINE
# ==========================================
def build_license_dataframe(input_file, output_csv):
    # Load JSONL into Pandas
    # lines=True tells pandas that each line is a separate JSON object
    print(f"Loading data from {input_file}...")
    df = pd.read_json(input_file, lines=True)

    # Apply the cleaning logic to the 'license' column
    print("Standardizing licenses...")
    df['license_cleaned'] = df['license'].apply(process_license_field)

    # HELPER COLUMN FOR GRAPHING:
    # Since some licenses are lists, we create a 'flat' version.
    # If it's a list, we join it with a comma. This makes it a string, 
    # which is required for most graphing libraries.
    df['license_flat'] = df['license_cleaned'].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x
    )

    # Save to CSV for future use
    df.to_csv(output_csv, index=False)
    print(f"DataFrame saved to {output_csv}")
    return df

In [ ]:
# Configuration
DATA_DIR = 'clean_license_data'
INPUT_JSONL = os.path.join(DATA_DIR, 'final_combined_licenses.jsonl')
OUTPUT_CSV = os.path.join(DATA_DIR, 'license_analysis.csv')
# Run the pipeline
df_licenses = build_license_dataframe(INPUT_JSONL, OUTPUT_CSV)

# --- Quick Analysis for Verification ---
print("\n" + "="*30)
print("QUICK DATASET SUMMARY")
print("="*30)
print(f"Total Models: {len(df_licenses)}")
print("\nTop 10 Licenses (Flat):")
print(df_licenses['license_flat'].value_counts().head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime # Added for timestamping

%matplotlib inline 

In [ ]:
def plot_license_distribution(csv_file, data_dir, top_n=15, save_file=True):
    # Load the data
    if not os.path.exists(csv_file):
        print(f"Error: {csv_file} not found.")
        return

    df = pd.read_csv(csv_file)

    # Prepare the data
    license_counts = df['license_flat'].value_counts().head(top_n)
    licenses = license_counts.index
    counts = license_counts.values

    # Create the Visualization
    plt.figure(figsize=(12, 8))

    bars = plt.barh(licenses, counts, color='skyblue', edgecolor='navy')

    # Final Polish
    plt.title(f'Top {top_n} Model Licenses in Dataset', fontsize=16, fontweight='bold')
    plt.xlabel('Number of Models', fontsize=12)
    plt.ylabel('License Name', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.gca().invert_yaxis()

    # Add number labels to bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.5, 
                    bar.get_y() + bar.get_height()/2, 
                    f'{int(width)}', 
                    va='center', 
                    fontweight='bold')

    plt.tight_layout()

    # --- NEW: Save functionality ---
    if save_file:
        # Generate timestamp: YYYYMMDD_HHMMSS (e.g., 20231027_143005)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"license_graph_{timestamp}.png"
        save_path = os.path.join(data_dir, filename)
        
        # bbox_inches='tight' prevents the labels from being cut off in the saved file
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Graph saved successfully to: {save_path}")

    plt.show()

In [ ]:
#Execution
# --- Configuration ---
DATA_DIR = 'clean_license_data'
CSV_FILE = os.path.join(DATA_DIR, 'license_analysis.csv')

# Run the plot - I added DATA_DIR as an argument so the function knows where to save
plot_license_distribution(CSV_FILE, DATA_DIR, top_n=15, save_file=True)

## Timeseries
We now need to understand how license releases are changing over time